In [4]:
import time
from enum import Enum
from typing import Optional, Callable, TypedDict, Literal
from langgraph.graph import StateGraph, START, END


# ==========================================
# 1. Circuit Breaker 共享类定义
# ==========================================
class CircuitState(Enum):
    CLOSED = "CLOSED"
    OPEN = "OPEN"
    HALF_OPEN = "HALF_OPEN"


class FakeClock:
    def __init__(self, start_time: float = 1000.0):
        self.now = start_time

    def time(self) -> float:
        return self.now

    def advance(self, seconds: float):
        self.now += seconds


class CircuitBreaker:
    def __init__(
        self,
        failure_threshold: int = 3,
        cooldown: float = 30.0,
        clock_fn: Callable[[], float] = time.time,
    ):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown
        self.clock_fn = clock_fn

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at: Optional[float] = None
        self.half_open_probe_in_flight = False

    def can_call(self) -> bool:
        now = self.clock_fn()

        if self.state == CircuitState.CLOSED:
            return True

        if self.state == CircuitState.OPEN:
            if self.opened_at is None:
                raise RuntimeError(
                    "[CircuitBreaker Error] Invalid state: OPEN but 'opened_at' is None."
                )

            elapsed = now - self.opened_at
            if elapsed < self.cooldown:
                return False

            print(
                f"[CircuitBreaker] ⏱️ Cooldown 结束 ({elapsed:.2f}s >= {self.cooldown}s)，状态转换: OPEN -> HALF_OPEN"
            )
            self.state = CircuitState.HALF_OPEN
            self.half_open_probe_in_flight = True
            return True

        if self.state == CircuitState.HALF_OPEN:
            if self.half_open_probe_in_flight:
                print(
                    "[CircuitBreaker] 🛡️ HALF_OPEN 状态已有试探请求在执行中，拦截并发请求 (Fast Fail)"
                )
                return False

            self.half_open_probe_in_flight = True
            return True

        return False

    def record_success(self):
        if self.state == CircuitState.HALF_OPEN:
            print(
                "[CircuitBreaker] 🟢 HALF_OPEN 试探成功！服务恢复，状态转换: HALF_OPEN -> CLOSED"
            )
        else:
            print("[CircuitBreaker] 🟢 请求成功，重置失败计数")

        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.opened_at = None
        self.half_open_probe_in_flight = False

    def record_failure(self):
        now = self.clock_fn()

        if self.state == CircuitState.HALF_OPEN:
            print(
                "[CircuitBreaker] 🚨 HALF_OPEN 试探失败！重新触发熔断，状态转换: HALF_OPEN -> OPEN"
            )
            self.state = CircuitState.OPEN
            self.opened_at = now
            self.half_open_probe_in_flight = False
            return

        if self.state == CircuitState.CLOSED:
            self.failure_count += 1
            print(
                f"[CircuitBreaker] ⚠️ 连续失败计次: {self.failure_count}/{self.failure_threshold}"
            )
            if self.failure_count >= self.failure_threshold:
                print(
                    f"[CircuitBreaker] 🚨 连续失败达阈值 ({self.failure_threshold})！状态转换: CLOSED -> OPEN"
                )
                self.state = CircuitState.OPEN
                self.opened_at = now
                self.half_open_probe_in_flight = False
            return

        if self.state == CircuitState.OPEN:
            print("[CircuitBreaker] ℹ️ 处于 OPEN 状态，忽略无效的 failure 记录")
            return


# 全局共享实例与测试 Mock 句柄
fake_clock = FakeClock(start_time=1000.0)
hr_api_breaker = CircuitBreaker(
    failure_threshold=3, cooldown=30.0, clock_fn=fake_clock.time
)
mock_api_should_succeed = True
mock_api_called = False  # 关键验证指标：确认 Fast Fail 时未真实调用 API


# ==========================================
# 2. Agent State 与 LangGraph 节点构建
# ==========================================
class WorkflowState(TypedDict):
    employee_id: str
    amount: float
    gate_action: Optional[Literal["ALLOW", "FAST_FAIL"]]
    api_status: Optional[int]
    result: Optional[str]


def check_circuit_breaker_node(state: WorkflowState):
    """门禁节点：只查共享 Breaker，不改写/读取 state 里的熔断数据"""
    if hr_api_breaker.can_call():
        print("[Gate Node] ✅ Circuit Breaker 放行 (ALLOW)")
        return {"gate_action": "ALLOW"}
    else:
        print("[Gate Node] ⛔ Circuit Breaker 拦截 (FAST_FAIL)")
        return {"gate_action": "FAST_FAIL"}


def fast_fail_fallback_node(state: WorkflowState):
    """熔断触发时的快速降级节点"""
    print("[Fallback Node] 🚀 Fast Fail 触发，立即返回降级结果 (不等待 API 超时)")
    return {"result": "FALLBACK: Downstream HR Service is temporarily unavaliable (Circuit Open)"}


def call_hr_api_node(state: WorkflowState):
    """实际执行下游调用的节点"""
    global mock_api_called
    mock_api_called = True

    if mock_api_should_succeed:
        print(f"[HR API] 200 OK - 成功发放薪资 {state['amount']} 至 Employee {state['employee_id']}")
        return {"api_status": 200}
    else:
        print(f"[HR API] 503 Service Unavailable - Downstream failure")
        return {"api_status": 503}


def record_success_node(state: WorkflowState):
    hr_api_breaker.record_success()
    return {"result": "SUCCESS: Salary transferred successfully."}


def record_failure_node(state: WorkflowState):
    hr_api_breaker.record_failure()
    return {"result": "FALLBACK: HR API execution failed, routing to manual review."}


# 路由判断逻辑
def route_gate(state: WorkflowState) -> Literal["call_hr_api", "fast_fail_fallback_node"]:
    if state["gate_action"] == "ALLOW":
        return "call_hr_api"
    return "fast_fail_fallback_node"


def route_api_result(state: WorkflowState) -> Literal["record_success_node", "record_failure_node"]:
    if state["api_status"] == 200:
        return "record_success_node"
    return "record_failure_node"


# 构建 Graph
builder = StateGraph(WorkflowState)
builder.add_node("check_circuit_breaker", check_circuit_breaker_node)
builder.add_node("fast_fail_fallback_node", fast_fail_fallback_node)
builder.add_node("call_hr_api", call_hr_api_node)
builder.add_node("record_success_node", record_success_node)
builder.add_node("record_failure_node", record_failure_node)

builder.add_edge(START, "check_circuit_breaker")
builder.add_conditional_edges(
    "check_circuit_breaker",
    route_gate,
    {
        "call_hr_api": "call_hr_api",
        "fast_fail_fallback_node": "fast_fail_fallback_node",
    },
)
builder.add_conditional_edges(
    "call_hr_api",
    route_api_result,
    {
        "record_success_node": "record_success_node",
        "record_failure_node": "record_failure_node",
    },
)
builder.add_edge("fast_fail_fallback_node", END)
builder.add_edge("record_success_node", END)
builder.add_edge("record_failure_node", END)

graph = builder.compile()


class FakeClock:
    def __init__(self, start_time: float = 1000.0):
        self.now = start_time

    def time(self) -> float:
        return self.now

    def advance(self, seconds: float):
        self.now += seconds


if __name__ == "__main__":
    fake_clock = FakeClock(start_time=1000.0)
    
    # 将 fake_clock.time 注入进 Circuit Breaker (阈值 2 次，Cooldown 30 秒)
    breaker = CircuitBreaker(failure_threshold=2, cooldown=30.0, clock_fn=fake_clock.time)

    print("--- 1. 触发熔断 (CLOSED -> OPEN) ---")
    breaker.record_failure()
    breaker.record_failure()
    print(f"Current State: {breaker.state.value}") # OPEN
    print(f"Can Call right now? {breaker.can_call()}") # False

    print("\n--- 2. 瞬间快进 10 秒 (Cooldown 未到) ---")
    fake_clock.advance(10.0)
    print(f"Can Call at +10s? {breaker.can_call()}") # False (Fast Fail)

    print("\n--- 3. 瞬间快进 21 秒 (累计 31s > 30s Cooldown) ---")
    fake_clock.advance(21.0)
    print(f"Can Call probe at +31s? {breaker.can_call()}") # True (转换为 HALF_OPEN)
    
    print("\n--- 4. 试探失败重置 OPEN ---")
    breaker.record_failure()
    print(f"Current State: {breaker.state.value}") # OPEN

--- 1. 触发熔断 (CLOSED -> OPEN) ---
[CircuitBreaker] ⚠️ 连续失败计次: 1/2
[CircuitBreaker] ⚠️ 连续失败计次: 2/2
[CircuitBreaker] 🚨 连续失败达阈值 (2)！状态转换: CLOSED -> OPEN
Current State: OPEN
Can Call right now? False

--- 2. 瞬间快进 10 秒 (Cooldown 未到) ---
Can Call at +10s? False

--- 3. 瞬间快进 21 秒 (累计 31s > 30s Cooldown) ---
[CircuitBreaker] ⏱️ Cooldown 结束 (31.00s >= 30.0s)，状态转换: OPEN -> HALF_OPEN
Can Call probe at +31s? True

--- 4. 试探失败重置 OPEN ---
[CircuitBreaker] 🚨 HALF_OPEN 试探失败！重新触发熔断，状态转换: HALF_OPEN -> OPEN
Current State: OPEN
